# チュートリアル 03: コンテキストとメモリ - パーソナライズされた旅行アシスタント

## 📚 学習目標
このノートブックを終える頃には、以下ができるようになります:
- スレッド(会話履歴)とメモリ(ユーザー知識)の違いを理解する
- 永続的なメモリを追加するカスタムContext Provider (`BaseContextProvider`) を作成する
- ユーザーの好みを自動的に抽出して保存する
- セッション間でユーザーの詳細を記憶するパーソナライズされたエージェントを構築する
- メモリの状態をシリアライズして復元する

## 🎯 重要な概念

### スレッド vs メモリ: 何が違うのか?

**スレッド (チュートリアル03より):**
- 会話履歴を保存する(「私はXと言った、あなたはYと言った」)
- 一時的: 単一の会話セッションに適している
- 例: 「ユーザーがパリについて尋ね、その後そこの天気について尋ねた」

**メモリ (このチュートリアル):**
- ユーザーに関する事実を保存する(「ユーザーはビーチバケーションを好む」)
- 永続的: セッションとスレッドを超えて存続する
- 例: 「ユーザーの名前はサラ、予算重視、ベジタリアン」

### Context Provider (`BaseContextProvider`)

**Context Provider**は以下を行うコンポーネントです:
1. 会話を**リッスン**する(`after_run()`メソッド経由)
2. 重要な情報を**抽出**する(ユーザーの好み、事実)
3. 各AI呼び出し前に**コンテキストを提供**する(`before_run()`メソッド経由)
4. セッション間で**永続化**する(`serialize()`メソッド経由)

```python
class CustomMemory(BaseContextProvider):
    async def after_run(self, *, agent, session, context, state):
        # エージェントが応答した後に呼び出される
        # context.input_messages から情報を抽出
    
    async def before_run(self, *, agent, session, context, state):
        # エージェントが応答する前に呼び出される
        # context.extend_instructions() で記憶されたコンテキストを提供
    
    def serialize(self):
        # メモリの状態を保存
```

---

## ステップ 1: セットアップとインポート

In [ ]:
import asyncio
import json
from collections.abc import MutableSequence, Sequence
from typing import Annotated, Any
from random import randint, choice

from agent_framework import (
    Agent,
    AgentSession,
    BaseContextProvider,
    ChatOptions,
    Message,
    SessionContext,
    SupportsAgentRun,
    SupportsChatGetResponse,
)
from agent_framework.azure import AzureOpenAIChatClient
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import DefaultAzureCredential
from azure.identity.aio import AzureCliCredential, DefaultAzureCredential

from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()
print("✅ インポート成功！")

In [ ]:
import os

mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === 認証方式の選択 ===
# True: APIキー認証, False: DefaultAzureCredential 認証
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    print("🔑 認証方式: APIキー認証")
else:
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 認証方式: DefaultAzureCredential")

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

# NOTE: Foundry の会話クライアントと、メモリ抽出用の structured output クライアントは分ける。
# Foundry 側は response_format が安定しないことがあるため、抽出は Azure OpenAI Chat を使う。
def create_azure_openai_chat_client(credential):
    return AzureAIAgentClient(
        credential=credential,
        project_endpoint=project_endpoint,
        model_deployment_name=model_deployment,
    )

def create_azure_openai_chat_client(credential=None):
    client_kwargs = {
        "endpoint": azure_openai_endpoint,
        "api_version": api_version,
        "deployment_name": azure_deployment,
    }
    if USE_KEY_AUTH:
        client_kwargs["api_key"] = azure_openai_key
    else:
        client_kwargs["credential"] = credential
    return AzureOpenAIChatClient(**client_kwargs)


def create_memory_extraction_client(credential=None):
    return create_azure_openai_chat_client(credential)


print(f"MCP Server URI: {mcp_server_uri}")
print(f"Foundry Project Endpoint: {project_endpoint}")
print(f"Foundry Model Deployment: {model_deployment}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## ステップ 2: 問題点 - メモリのないエージェント

まず、制限事項を見てみましょう: スレッドがあっても、エージェントはセッション間でユーザーの詳細を覚えていません。

In [ ]:
async def agent_without_memory():
    """
    セッションは会話を記憶するが、ユーザーに関する事実は記憶しないことを実証します。
    """
    print("=== メモリなしエージェント (セッションのみ) ===")
    print("セッションは会話を記憶しますが、ユーザーに関する事実は記憶しません。\n")

    if USE_KEY_AUTH:
        foundry_client = create_azure_openai_chat_client()
        async with Agent(
            client=foundry_client,
            name="BasicTravelAgent",
            instructions="あなたは旅行アシスタントです。ユーザーの旅行計画を支援してください。",
        ) as agent:
            session1 = agent.create_session()

            response1 = await agent.run("私の名前はSarahで、ビーチバケーションが大好きです。", session=session1)
            print("ユーザー: 私の名前はSarahで、ビーチバケーションが大好きです。")
            print(f"エージェント: {response1.text}\n")

            print("--- セッション 2 (新しいセッション) ---")
            session2 = agent.create_session()

            response2 = await agent.run("目的地を推奨していただけますか？", session=session2)
            print("ユーザー: 目的地を推奨していただけますか？")
            print(f"エージェント: {response2.text}\n")

            print("❌ 問題: エージェントはサラの名前とビーチの好みを忘れました!")
            print("   たとえsession1を再利用しても、好みは事実として「記憶」されません。\n")
            return

    async with DefaultAzureCredential() as credential:
        foundry_client = create_azure_openai_chat_client(credential)
        async with Agent(
            client=foundry_client,
            name="BasicTravelAgent",
            instructions="あなたは旅行アシスタントです。ユーザーの旅行計画を支援してください。",
        ) as agent:
            session1 = agent.create_session()

            response1 = await agent.run("私の名前はSarahで、ビーチバケーションが大好きです。", session=session1)
            print("ユーザー: 私の名前はSarahで、ビーチバケーションが大好きです。")
            print(f"エージェント: {response1.text}\n")

            print("--- セッション 2 (新しいセッション) ---")
            session2 = agent.create_session()

            response2 = await agent.run("目的地を推奨していただけますか？", session=session2)
            print("ユーザー: 目的地を推奨していただけますか？")
            print(f"エージェント: {response2.text}\n")

            print("❌ 問題: エージェントはサラの名前とビーチの好みを忘れました!")
            print("   たとえsession1を再利用しても、好みは事実として「記憶」されません。\n")

await agent_without_memory()

## ステップ 3: ユーザープロファイルモデルの定義

まず、ユーザーについて記憶したいことを定義しましょう。

In [ ]:
class TravelProfile(BaseModel):
    """ユーザーの旅行の好みとプロファイル。"""

    name: str | None = Field(None, description="ユーザーの名前")
    preferred_destinations: list[str] = Field(
        default_factory=list,
        description="ユーザーが好む目的地のタイプ（例: ビーチ、山、都市 など）",
    )
    budget_level: str | None = Field(
        None,
        description="予算感（例: 節約/中程度/贅沢）",
    )
    dietary_restrictions: list[str] = Field(
        default_factory=list,
        description="食事制限（例: ベジタリアン、ヴィーガン、ハラール、コーシャ など）",
    )
    interests: list[str] = Field(
        default_factory=list,
        description="旅行の興味関心（例: 歴史、食、冒険、リラックス など）",
    )


print("✅ TravelProfileモデル定義完了")
print("\nプロファイル例:")
example = TravelProfile(
    name="Sarah",
    preferred_destinations=["ビーチ", "南国"],
    budget_level="中程度",
    dietary_restrictions=["ベジタリアン"],
    interests=["リラックス", "食"],
)
print(example.model_dump_json(indent=2))

## ステップ 4: メモリ付きContext Providerの作成

それでは、メモリコンポーネントを構築しましょう!

In [ ]:
import re

class TravelMemory(BaseContextProvider):
    """
    ユーザーの旅行の好みを記憶するContext Provider。

    公式サンプル simple_context_provider.py に準拠した実装:
    1. 会話をリッスンし、ユーザー情報を抽出する (after_run)
    2. 各エージェント応答前に記憶されたコンテキストを提供する (before_run)
    3. セッション間でユーザープロファイルを永続化する
    """

    def __init__(
        self,
        chat_client: SupportsChatGetResponse,
        profile: TravelProfile | None = None,
        source_id: str = "travel_memory",
        **kwargs: Any,
    ):
        super().__init__(source_id)
        self._chat_client = chat_client

        if profile:
            self.profile = profile
        elif kwargs:
            self.profile = TravelProfile.model_validate(kwargs)
        else:
            self.profile = TravelProfile()

    @staticmethod
    def _extract_json(text: str) -> str:
        md_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
        if md_match:
            return md_match.group(1)
        brace_match = re.search(r'\{.*\}', text, re.DOTALL)
        if brace_match:
            return brace_match.group(0)
        return ""

    @staticmethod
    def _has_updates(profile: TravelProfile) -> bool:
        return any(
            [
                bool(profile.name),
                bool(profile.preferred_destinations),
                bool(profile.budget_level),
                bool(profile.dietary_restrictions),
                bool(profile.interests),
            ]
        )

    async def after_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        """直近のユーザー入力から旅行プロファイルを抽出する。"""
        user_messages = [
            msg for msg in context.input_messages
            if hasattr(msg, "role") and msg.role == "user"
        ]
        if not user_messages:
            return

        try:
            result = await self._chat_client.get_response(
                messages=user_messages,
                options={
                    "instructions": """
                    次のユーザー入力から、旅行プロファイルの更新情報だけを抽出してください。
                    - 抽出対象: 名前、目的地の好み、予算感、食事制限、興味関心
                    - 明示的に言及されているフィールドのみ抽出すること
                    - 言及のないフィールドは null または空リストにすること
                    - 新しいプロフィール情報が一切ない場合は、空のJSONオブジェクト {} を返すこと
                    - 必ずJSONのみを返し、説明文やマークダウンは付けないこと
                    """,
                    "response_format": TravelProfile,
                },
            )
        except Exception:
            return

        extracted: TravelProfile | None = None
        try:
            extracted = result.value
        except Exception:
            raw_json = self._extract_json(result.text)
            if not raw_json or raw_json == "{}":
                return
            try:
                extracted = TravelProfile.model_validate_json(raw_json)
            except Exception:
                return

        if not extracted or not self._has_updates(extracted):
            return

        if extracted.name:
            self.profile.name = extracted.name

        if extracted.preferred_destinations:
            for dest in extracted.preferred_destinations:
                if dest not in self.profile.preferred_destinations:
                    self.profile.preferred_destinations.append(dest)

        if extracted.budget_level:
            self.profile.budget_level = extracted.budget_level

        if extracted.dietary_restrictions:
            for restriction in extracted.dietary_restrictions:
                if restriction not in self.profile.dietary_restrictions:
                    self.profile.dietary_restrictions.append(restriction)

        if extracted.interests:
            for interest in extracted.interests:
                if interest not in self.profile.interests:
                    self.profile.interests.append(interest)

        print(f"  📝 [Memory] 抽出結果: {extracted.model_dump()}")

    async def before_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        memory_lines: list[str] = []

        if self.profile.name:
            memory_lines.append(f"ユーザーの名前: {self.profile.name}")
        if self.profile.preferred_destinations:
            memory_lines.append(f"好みの目的地: {', '.join(self.profile.preferred_destinations)}")
        if self.profile.budget_level:
            memory_lines.append(f"予算感: {self.profile.budget_level}")
        if self.profile.dietary_restrictions:
            memory_lines.append(f"食事制限: {', '.join(self.profile.dietary_restrictions)}")
        if self.profile.interests:
            memory_lines.append(f"興味関心: {', '.join(self.profile.interests)}")

        if not memory_lines:
            return

        memory_context = (
            "このユーザーについて覚えていること: "
            + ", ".join(memory_lines)
            + "。必ずこの情報を考慮して、名前を使い、パーソナライズされた提案を行うこと。"
        )

        print(f"🔍 [Context Provider] メモリを注入: {self.profile.name}")
        context.extend_instructions(self.source_id, memory_context)

    def serialize(self) -> str:
        return self.profile.model_dump_json()


print("✅ TravelMemory コンテキストプロバイダーを作成しました！")

## ステップ 5: メモリ付きエージェント - 最初の会話

メモリの動作を見てみましょう!

In [ ]:
async def agent_with_memory_first_session():
    """
    最初のセッション: エージェントがユーザーについて学習します。
    """
    print("=== メモリ付きエージェント - ユーザーについて学習中 ===")

    if USE_KEY_AUTH:
        foundry_client = create_azure_openai_chat_client()
        memory = TravelMemory(foundry_client)

        async with Agent(
            client=foundry_client,
            name="MemoryEnabledTravelAgent",
            instructions="""
            あなたはパーソナライズされた旅行アシスタントです。
            ユーザーの名前が分かる場合は必ず名前で呼びかけてください。
            ユーザーの好みに基づいて提案を調整してください。
            """,
            context_providers=[memory],
        ) as agent:
            session = agent.create_session()

            query1 = "こんにちは！私の名前はSarahで、ビーチでの休暇が大好きです。"
            print(f"\nユーザー: {query1}")
            response1 = await agent.run(query1, session=session)
            print(f"エージェント: {response1.text}")

            query2 = "予算は中くらいで、高すぎない範囲がいいです。"
            print(f"\nユーザー: {query2}")
            response2 = await agent.run(query2, session=session)
            print(f"エージェント: {response2.text}")

            query3 = "それと、私はベジタリアンなので食事の選択肢が重要です。"
            print(f"\nユーザー: {query3}")
            response3 = await agent.run(query3, session=session)
            print(f"エージェント: {response3.text}")

            print("\n" + "=" * 60)
            print("メモリが学習した内容:")
            print("=" * 60)
            print(f"名前: {memory.profile.name}")
            print(f"好みの目的地: {memory.profile.preferred_destinations}")
            print(f"予算レベル: {memory.profile.budget_level}")
            print(f"食事制限: {memory.profile.dietary_restrictions}")
            print(f"興味: {memory.profile.interests}")

            saved_memory = memory.serialize()
            print("\n💾 メモリを永続化のためにシリアライズしました:")
            print(saved_memory)
            return saved_memory

    async with DefaultAzureCredential() as credential:
        foundry_client = create_azure_openai_chat_client(credential)
        memory = TravelMemory(foundry_client)

        async with Agent(
            client=foundry_client,
            name="MemoryEnabledTravelAgent",
            instructions="""
            あなたはパーソナライズされた旅行アシスタントです。
            ユーザーの名前が分かる場合は必ず名前で呼びかけてください。
            ユーザーの好みに基づいて提案を調整してください。
            """,
            context_providers=[memory],
        ) as agent:
            session = agent.create_session()

            query1 = "こんにちは！私の名前はSarahで、ビーチでの休暇が大好きです。"
            print(f"\nユーザー: {query1}")
            response1 = await agent.run(query1, session=session)
            print(f"エージェント: {response1.text}")

            query2 = "予算は中くらいで、高すぎない範囲がいいです。"
            print(f"\nユーザー: {query2}")
            response2 = await agent.run(query2, session=session)
            print(f"エージェント: {response2.text}")

            query3 = "それと、私はベジタリアンなので食事の選択肢が重要です。"
            print(f"\nユーザー: {query3}")
            response3 = await agent.run(query3, session=session)
            print(f"エージェント: {response3.text}")

            print("\n" + "=" * 60)
            print("メモリが学習した内容:")
            print("=" * 60)
            print(f"名前: {memory.profile.name}")
            print(f"好みの目的地: {memory.profile.preferred_destinations}")
            print(f"予算レベル: {memory.profile.budget_level}")
            print(f"食事制限: {memory.profile.dietary_restrictions}")
            print(f"興味: {memory.profile.interests}")

            saved_memory = memory.serialize()
            print("\n💾 メモリを永続化のためにシリアライズしました:")
            print(saved_memory)
            return saved_memory


saved_memory = await agent_with_memory_first_session()

In [ ]:
saved_memory

## ステップ 6: 保存されたメモリで再開 - インタラクティブチャット

それでは、保存されたメモリを使って新しいエージェントを作成しましょう - 新しいセッションをシミュレートします。

**注意**: Context Providerが正しく動作するように改善しました。以下のセルを実行する前に、**セル10 (TravelMemory クラス定義)を再実行**してください。

In [ ]:
async def agent_with_restored_memory(saved_memory_json: str):
    """
    新しいセッション: エージェントがメモリを復元し、ユーザーを覚えています!
    """
    print("\n=== メモリ付きエージェント - 新しいセッション (復元されたメモリ) ===")
    print("保存されたメモリを使って完全に新しいセッションをシミュレート中...\n")

    profile_dict = json.loads(saved_memory_json)
    restored_profile = TravelProfile.model_validate(profile_dict)

    print("✅ 復元されたプロファイル:")
    print(f"   名前: {restored_profile.name}")
    print(f"   好み: {restored_profile.preferred_destinations}")
    print(f"   予算: {restored_profile.budget_level}\n")

    if USE_KEY_AUTH:
        foundry_client = create_azure_openai_chat_client()
        memory = TravelMemory(foundry_client, profile=restored_profile)

        async with Agent(
            client=foundry_client,
            name="MemoryEnabledTravelAgent_Session2",
            instructions="""
            あなたはパーソナライズされた旅行アシスタントです。
            ユーザーの名前が分かる場合は必ず名前で呼びかけてください。
            ユーザーの好みに基づいて提案を調整してください。
            """,
            context_providers=[memory],
        ) as agent:
            session = agent.create_session()
            query = "次の旅行先をおすすめしてくれますか？"
            print(f"ユーザー: {query}")
            response = await agent.run(query, session=session)
            print(f"エージェント: {response.text}\n")

            print("=" * 60)
            print("🎉 成功!")
            print("=" * 60)
            print("エージェントが次のことを行ったことに注目してください:")
            print("1. サラの名前を使用した")
            print("2. ビーチの目的地を推奨した")
            print("3. 中程度の予算を考慮した")
            print("4. ベジタリアンの食事オプションに言及した")
            print("\nすべて異なるセッションからです!")
            return

    async with DefaultAzureCredential() as credential:
        foundry_client = create_azure_openai_chat_client(credential)
        memory = TravelMemory(foundry_client, profile=restored_profile)

        async with Agent(
            client=foundry_client,
            name="MemoryEnabledTravelAgent_Session2",
            instructions="""
            あなたはパーソナライズされた旅行アシスタントです。
            ユーザーの名前が分かる場合は必ず名前で呼びかけてください。
            ユーザーの好みに基づいて提案を調整してください。
            """,
            context_providers=[memory],
        ) as agent:
            session = agent.create_session()
            query = "次の旅行先をおすすめしてくれますか？"
            print(f"ユーザー: {query}")
            response = await agent.run(query, session=session)
            print(f"エージェント: {response.text}\n")

            print("=" * 60)
            print("🎉 成功!")
            print("=" * 60)
            print("エージェントが次のことを行ったことに注目してください:")
            print("1. サラの名前を使用した")
            print("2. ビーチの目的地を推奨した")
            print("3. 中程度の予算を考慮した")
            print("4. ベジタリアンの食事オプションに言及した")
            print("\nすべて異なるセッションからです!")


await agent_with_restored_memory(saved_memory)

## ステップ 7: 時間経過によるメモリの更新

メモリはユーザーがより多くの情報を共有するにつれて進化できます。

In [ ]:
async def memory_evolution():
    """
    メモリが時間とともに成長し、更新される様子を実証します。
    """
    print("=== 時間経過によるメモリの進化 ===")

    async with DefaultAzureCredential() as credential:
        conversation_client = create_azure_openai_chat_client(credential)
        memory_client = create_memory_extraction_client(credential)
        memory = TravelMemory(memory_client)

        async with Agent(
            client=conversation_client,
            name="EvolvingMemoryAgent",
            instructions="あなたは親切な旅行アシスタントです。",
            context_providers=[memory],
        ) as agent:
            session = agent.create_session()

            print("\n--- 1週目: 最初の旅行 ---")
            await agent.run("私はTomです。山歩き（ハイキング）が好きです。", session=session)
            print(f"メモリ: {memory.profile.model_dump()}\n")

            print("--- 2週目: より多くを発見 ---")
            await agent.run("歴史や文化的な名所も大好きです。", session=session)
            print(f"メモリ: {memory.profile.model_dump()}\n")

            print("--- 3週目: 予算計画 ---")
            await agent.run("今回は少し贅沢な体験をしたいです。", session=session)
            print(f"メモリ: {memory.profile.model_dump()}\n")

            print("--- 4週目: 視野を広げる ---")
            await agent.run("今回は都市での短い旅行（シティブレイク）も試してみたいです。", session=session)
            print(f"メモリ: {memory.profile.model_dump()}\n")

            print("🎯 メモリが基本的なものから豊かなユーザープロファイルに進化しました!")


await memory_evolution()

## ステップ 8: 実践的な例 - マルチセッション計画

実世界のシナリオ: ユーザーが複数のセッションにわたって旅行を計画します。

In [ ]:
# 以前のチュートリアルからのツールを追加
def get_weather(
    location: Annotated[str, Field(description="都市名または国名")],
) -> str:
    """指定された場所の現在の天気を返します。"""
    conditions = ["晴れ", "晴れ時々曇り", "曇り", "雨"]
    temp = randint(15, 32)
    return f"{location}の天気: {choice(conditions)}, {temp}°C"


def get_attractions(
    location: Annotated[str, Field(description="都市名または国名")],
) -> str:
    """目的地の主な観光スポットを返します。"""
    attractions = {
        "Bali": "ビーチ、ライステラス、寺院、サーフィン、ヨガリトリート",
        "Thailand": "バンコクの寺院、ビーチ、マーケット、屋台料理、島々",
        "Maldives": "ビーチ、ダイビング、高級リゾート、ウォータースポーツ",
    }
    for city, attrs in attractions.items():
        if city.lower() in location.lower():
            return f"{location}の主な観光スポット: {attrs}"
    return f"{location}の人気観光スポット: さまざまな観光名所"


print("✅ ツール定義完了")

In [ ]:
async def multi_session_planning():
    """
    永続的なメモリを使った現実的なマルチセッション旅行計画。
    """
    print("=== マルチセッション旅行計画 ===")

    print("\n--- セッション 1: 初期探索 ---")
    async with DefaultAzureCredential() as credential:
        conversation_client = create_azure_openai_chat_client(credential)
        memory_client = create_memory_extraction_client(credential)
        memory = TravelMemory(memory_client)

        async with Agent(
            client=conversation_client,
            name="TravelPlanner",
            instructions="""
            あなたはパーソナライズされた旅行プランナーです。
            ユーザーについて知っていることを活用して、カスタマイズされた提案を行ってください。
            """,
            tools=[get_weather, get_attractions],
            context_providers=[memory],
        ) as agent:
            session1 = agent.create_session()

            query1 = "こんにちは、私はEmmaです。リラックスできるビーチでの休暇を計画したいです。私はベジタリアンです。"
            print(f"ユーザー: {query1}")
            response = await agent.run(query1, session=session1)
            print(f"エージェント: {response.text}\n")

        session1_memory = memory.serialize()
        print("💾 セッション1の後にメモリを保存しました")
        print(f"メモリの状態: {json.loads(session1_memory)}\n")

        print("--- セッション 2: 目的地の調査 (3日後) ---")
        conversation_client = create_azure_openai_chat_client(credential)
        memory_client = create_memory_extraction_client(credential)
        profile = TravelProfile.model_validate(json.loads(session1_memory))
        memory = TravelMemory(memory_client, profile=profile)
        print(f"✅ メモリを復元: 名前={profile.name}, 好み={profile.preferred_destinations}")

        async with Agent(
            client=conversation_client,
            name="TravelPlanner",
            instructions="""
            あなたはパーソナライズされた旅行プランナーです。
            ユーザーについて知っていることを活用して、カスタマイズされた提案を行ってください。
            """,
            tools=[get_weather, get_attractions],
            context_providers=[memory],
        ) as agent:
            session2 = agent.create_session()

            query2 = "バリ島かモルディブで迷っています。天気はどんな感じですか？"
            print(f"\nユーザー: {query2}")
            response = await agent.run(query2, session=session2)
            print(f"エージェント: {response.text}\n")

            query3 = "予算は中くらいです。どちらがおすすめですか？"
            print(f"ユーザー: {query3}")
            response = await agent.run(query3, session=session2)
            print(f"エージェント: {response.text}\n")

        session2_memory = memory.serialize()
        print("💾 セッション2の後にメモリを保存しました")
        print(f"メモリの状態: {json.loads(session2_memory)}\n")

        print("--- セッション 3: 最終決定 (1週間後) ---")
        conversation_client = create_azure_openai_chat_client(credential)
        memory_client = create_memory_extraction_client(credential)
        profile = TravelProfile.model_validate(json.loads(session2_memory))
        memory = TravelMemory(memory_client, profile=profile)
        print(f"✅ メモリを復元: 名前={profile.name}, 予算={profile.budget_level}")

        async with Agent(
            client=conversation_client,
            name="TravelPlanner",
            instructions="""
            あなたはパーソナライズされた旅行プランナーです。
            ユーザーについて知っていることを活用して、カスタマイズされた提案を行ってください。
            """,
            tools=[get_weather, get_attractions],
            context_providers=[memory],
        ) as agent:
            session3 = agent.create_session()

            query4 = "バリ島に決めました！絶対に外せない観光スポットはどこですか？"
            print(f"\nユーザー: {query4}")
            response = await agent.run(query4, session=session3)
            print(f"エージェント: {response.text}\n")

    print("=" * 60)
    print("🎉 マルチセッション計画完了!")
    print("=" * 60)
    print("注目:")
    print("- 3つの異なるセッション (数日間隔)")
    print("- 3つの異なるセッション (別々の会話)")
    print("- エージェントが記憶: エマの名前、ビーチの好み、ベジタリアン、予算")
    print("- メモリがすべてのセッションにわたって永続化!")


await multi_session_planning()

## 🔍 Context Providerの理解

### Context Providerのライフサイクル

```python
# 1. ユーザーがメッセージを送信
await agent.run("ビーチが大好き", session=session)

# 2. AI呼び出し前: before_run()が呼び出される
await memory.before_run(agent=agent, session=session, context=context, state=state)
# context.extend_instructions() で追加指示を注入

# 3. AIが拡張された指示を受け取る
# オリジナル: "あなたは旅行アシスタントです"
# 拡張後: "あなたは旅行アシスタントです。ユーザーが好む目的地: ビーチ。"

# 4. AIがパーソナライズされた回答で応答

# 5. AI呼び出し後: after_run()が呼び出される
await memory.after_run(agent=agent, session=session, context=context, state=state)
# context.input_messages からユーザーメッセージを取得
# 抽出: ビーチの好み -> メモリに保存
```

### Context Providerが強力な理由

- **アプリケーションが状態を管理** - コンテキストプロバイダーはアプリケーションが永続化したデータを使用する
- **エージェントはステートレス** - エージェント定義は一時的だが、データは永続化される
- **柔軟なコンテキスト注入** - `context.extend_instructions()` / `context.extend_messages()` で必要なだけコンテキストを追加できる
- **デバッグ可能** - どのコンテキストが注入されたか確認できる

### 実運用での利用例

- ユーザープロファイル (名前、好み、権限)
- 会話履歴の要約
- 最近の操作ログ
- ビジネスルールやポリシー
- 共有アプリ状態 (カート、選択中のプラン等)

## 📝 重要なポイント

### 学んだこと

1. **スレッド ≠ メモリ**
   - スレッド: 会話履歴 (一時的)
   - メモリ: ユーザー知識 (永続的)

2. **Context Provider (`BaseContextProvider`)**
   - 会話をリッスン (`after_run()`)
   - コンテキストを提供 (`before_run()`)
   - 状態を永続化 (`serialize()`)

3. **パーソナライゼーション**
   - エージェントがセッション間でユーザーを記憶
   - 好みに基づいたカスタマイズされた推奨
   - 時間とともに理解が進化

4. **実践的なパターン**
   - メモリをデータベースに保存
   - セッション開始時にメモリを復元
   - ユーザーがより多くを共有するにつれてメモリを更新

### ベストプラクティス

1. 構造化されたメモリには**Pydanticモデルを使用**
2. データ損失を避けるため**頻繁にシリアライズ**
3. 保存前に**抽出されたデータを検証**
4. 最良の結果のため**スレッドと組み合わせる**
5. 抽出時の**エラーを適切に処理**

### 本番環境のパターン

```python
# データベース統合
class UserMemoryService:
    async def load_memory(self, user_id: str) -> TravelProfile:
        data = await db.get_user_profile(user_id)
        return TravelProfile.model_validate(data)
    
    async def save_memory(self, user_id: str, memory: TravelMemory):
        await db.update_user_profile(user_id, memory.serialize())

# 使用法
profile = await memory_service.load_memory(user_id)
memory = TravelMemory(chat_client, profile=profile)

# ... エージェントを使用 ...

await memory_service.save_memory(user_id, memory)
```

## 💪 練習問題

1. **TravelProfileの拡張** - preferred_activities、travel_companions、accessibility_needsなどのフィールドを追加
2. **マルチユーザーメモリ** - 複数のユーザーを処理するメモリシステムを作成
3. **メモリの減衰** - 時間ベースのメモリの関連性を実装 (古い好みの重要度が低くなる)
4. **矛盾の解決** - 矛盾する好みを処理 (「寒いのが嫌いと言ったのに、今はアラスカに行きたい?」)

In [ ]:
# 練習問題: アクティビティの好みを含む拡張プロファイルを作成

class EnhancedTravelProfile(BaseModel):
    """より多くの詳細を含む拡張旅行プロファイル。"""
    # ここにコードを記述 - さらにフィールドを追加!
    pass

# この拡張プロファイルを使用するコンテキストプロバイダーを作成
# さまざまなユーザー入力でテスト

## 🚀 次のステップ

素晴らしい! 永続的なメモリとパーソナライゼーションをマスターしました。

---

### クイックリファレンス

**Context Providerの作成:**
```python
class MyMemory(BaseContextProvider):
    async def after_run(self, *, agent, session, context, state):
        # context.input_messages から情報を抽出して保存
        pass
    
    async def before_run(self, *, agent, session, context, state):
        # コンテキストを注入
        context.extend_instructions(self.source_id, "ユーザーの好み: ...")
```

**エージェントでの使用:**
```python
memory = MyMemory(chat_client, source_id="my_memory")
agent = Agent(..., context_providers=[memory])
```

**メモリの永続化:**
```python
saved = memory.serialize()
# ... データベースに保存 ...
# ... 後で ...
restored = MyMemory(chat_client, **json.loads(saved))
```

---

# 🧠 ボーナス: Azure AI Foundry Memory Store によるメモリ永続化

## 概要

前のセクションでは `BaseContextProvider` を継承してカスタムメモリを自作しました。
ここでは、**Azure AI Foundry が提供するマネージドメモリストア** (`FoundryMemoryProvider`) を使って、
メモリの保存・検索・更新を **サーバーサイドで自動的に** 行う方法をテストします。

### FoundryMemoryProvider の特長

| 機能 | 説明 |
|---|---|
| **ユーザープロファイル自動抽出** | 会話からユーザー情報を自動的に抽出・蓄積 |
| **セマンティック検索** | 入力メッセージに関連するメモリを自動検索して注入 |
| **スコープ分離** | `scope` パラメータでユーザーごとにメモリを分離 |
| **遅延バッチ更新** | `update_delay` でコスト最適化（デフォルト5分） |

### 前提条件

- Azure AI Foundry プロジェクトが作成済み
- チャットモデル（gpt-4.1-mini 等）がデプロイ済み
- **埋め込みモデル**（text-embedding-3-small 等）がデプロイ済み
- `az login` 済み

### 公式サンプル

`agent-framework-python-1.0.0rc2/python/samples/02-agents/context_providers/azure_ai_foundry_memory.py`

## ステップ B1: Foundry Memory 用の追加インポートと設定

In [ ]:
from datetime import datetime, timezone

from agent_framework import InMemoryHistoryProvider
from agent_framework.azure import AzureOpenAIResponsesClient, FoundryMemoryProvider
from azure.ai.projects.aio import AIProjectClient
from azure.ai.projects.models import (
    MemoryStoreDefaultDefinition,
    MemoryStoreDefaultOptions,
)

# デプロイ名の設定 (.env から読み込み)
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME", "text-embedding-3-small")
responses_deployment = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME") or model_deployment

print("✅ Foundry Memory 用インポート完了")
print(f"   Project Endpoint: {project_endpoint}")
print(f"   Chat/Responses Model: {responses_deployment}")
print(f"   Embedding Model: {embedding_deployment}")

if not os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"):
    print("ℹ️ AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME が未設定のため、AZURE_AI_MODEL_DEPLOYMENT_NAME を使用します。")

## ステップ B2: メモリストアの作成

Azure AI Foundry にメモリストアを作成します。  
- `user_profile_enabled=True`: ユーザープロファイルの自動抽出を有効化  
- `chat_summary_enabled=False`: チャット要約は無効（デモのため）

メモリストアは名前がユニークである必要があるため、日時付きの名前を生成します。

In [ ]:
embedding_deployment

In [ ]:
project_endpoint = "https://foundry-sweden-central-hana1.services.ai.azure.com/api/projects/proj-default"

In [ ]:
# メモリストアの作成
memory_store_name = f"workshop_memory_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME", "text-embedding-3-small")
responses_deployment = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME") or model_deployment

# メモリストアオプション
options = MemoryStoreDefaultOptions(
    chat_summary_enabled=False,       # チャット要約は無効
    user_profile_enabled=True,        # ユーザープロファイル抽出を有効化
    user_profile_details="旅行の好み、食事制限、予算感、興味関心など旅行に関連する情報を抽出すること。"
    "年齢、金融情報、正確な住所、認証情報などの機密データは避けること。",
)

memory_store_definition = MemoryStoreDefaultDefinition(
    chat_model=responses_deployment,
    embedding_model=embedding_deployment,
    options=options,
)

# AIProjectClient を使ってメモリストアを作成
async with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client,
):
    print(f"メモリストア '{memory_store_name}' を作成中...")
    memory_store = await project_client.memory_stores.create(
        name=memory_store_name,
        description="Agent Framework Workshop: FoundryMemoryProvider テスト用",
        definition=memory_store_definition,
    )

    print("✅ メモリストア作成完了!")
    print(f"   名前: {memory_store.name}")
    print(f"   ID: {memory_store.id}")
    print(f"   説明: {memory_store.description}")

## ステップ B3: FoundryMemoryProvider を使ったエージェントの実行

公式サンプルと同じパターンで、`FoundryMemoryProvider` をエージェントに組み込みます。

**ポイント:**
- `scope="user_123"` — ユーザーIDでメモリをスコープ分離
- `update_delay=0` — デモ用に即座にメモリを更新（本番では300秒推奨）
- `InMemoryHistoryProvider(load_messages=False)` — 会話履歴をあえてロードしない
  - これにより、エージェントがFoundryメモリだけに頼って応答することを確認できる
- `default_options={"store": False}` — サービス側にも会話履歴を保存しない

In [ ]:
async def test_foundry_memory():
    """
    FoundryMemoryProvider を使ったセマンティックメモリのテスト。
    公式サンプル azure_ai_foundry_memory.py に準拠。
    """
    async with (
        DefaultAzureCredential() as credential,
        AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client,
    ):
        # チャットクライアントを作成（deployment_name を明示指定）
        client = AzureOpenAIResponsesClient(
            project_client=project_client,
            deployment_name=responses_deployment,
        )

        # FoundryMemoryProvider を作成
        memory_provider = FoundryMemoryProvider(
            project_client=project_client,
            memory_store_name=memory_store_name,
            scope="user_123",    # ユーザーIDでメモリをスコープ分離
            update_delay=0,      # デモ用: 即座にメモリ更新 (本番では300秒推奨)
        )

        # メモリ付きエージェントを作成
        async with Agent(
            name="FoundryMemoryAgent",
            client=client,
            instructions="""あなたは旅行に詳しい親切なアシスタントです。
                過去の会話からのメモリが自動的に提供されます。
                ユーザーの名前が分かる場合は必ず名前で呼びかけてください。
                ユーザーの好みに基づいてパーソナライズされた提案を行ってください。""",
            context_providers=[
                memory_provider,
                InMemoryHistoryProvider(load_messages=False),  # 履歴ロード無効（メモリのみをテスト）
            ],
            default_options={"store": False},  # サービス側にも会話履歴を保存しない
        ) as agent:
            session = agent.create_session()

            # === 会話 1: ユーザーの好みを伝える ===
            print("=" * 60)
            print("=== 会話 1: ユーザーの好みを伝える ===")
            print("=" * 60)
            query1 = "私の名前は Haruka です。ビーチリゾートが大好きで、ベジタリアンです。"
            print(f"ユーザー: {query1}")
            result1 = await agent.run(query1, session=session)
            print(f"エージェント: {result1.text}\n")

            # メモリが処理されるのを待つ
            print("⏳ メモリがストアに保存されるのを待機中 (8秒)...")
            await asyncio.sleep(8)

            # === 会話 2: メモリからの想起をテスト ===
            print("=" * 60)
            print("=== 会話 2: メモリからの想起をテスト ===")
            print("=" * 60)
            query2 = "次の旅行先でおすすめのレストランを教えてください。"
            print(f"ユーザー: {query2}")
            result2 = await agent.run(query2, session=session)
            print(f"エージェント: {result2.text}\n")

            # === 会話 3: メモリの内容を確認 ===
            print("=" * 60)
            print("=== 会話 3: メモリの内容を確認 ===")
            print("=" * 60)
            query3 = "私について覚えていることを教えてください。"
            print(f"ユーザー: {query3}")
            result3 = await agent.run(query3, session=session)
            print(f"エージェント: {result3.text}\n")

        # === メモリストアの中身を直接確認 ===
        print("=" * 60)
        print("📦 Foundry Memory Store に保存されたメモリ:")
        print("=" * 60)
        res = await project_client.memory_stores.search_memories(
            name=memory_store_name, scope="user_123"
        )
        for i, memory in enumerate(res.memories, 1):
            print(f"  {i}. {memory.memory_item.content}")

        if not res.memories:
            print("  (まだメモリが保存されていません)")


await test_foundry_memory()

## ステップ B4: クリーンアップ — メモリストアの削除

テスト完了後、メモリストアを削除してリソースをクリーンアップします。  
**注意**: 削除すると保存されたメモリはすべて失われます。

In [ ]:
# メモリストアの削除（クリーンアップ）
async with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client,
):
    await project_client.memory_stores.delete(memory_store_name)
    print(f"🗑️ メモリストア '{memory_store_name}' を削除しました。")

## 🔍 FoundryMemoryProvider vs カスタム TravelMemory の比較

| 観点 | カスタム TravelMemory (前半) | FoundryMemoryProvider (ボーナス) |
|---|---|---|
| **メモリ保存先** | アプリケーション側（変数 / JSON） | Azure AI Foundry サーバーサイド |
| **情報抽出** | 自前の LLM 呼び出しで構造化抽出 | Foundry が自動的にプロファイル抽出 |
| **検索方式** | なし（全メモリを常に注入） | セマンティック検索（関連メモリのみ取得） |
| **スケーラビリティ** | DB 連携が必要 | マネージドサービスで自動スケール |
| **コスト** | LLM 呼び出し × 2（抽出 + 応答） | Foundry Memory API 使用料 |
| **カスタマイズ性** | 完全に自由 | Foundry の仕様に準拠 |
| **実装の手間** | 多い（自分で全部書く） | 少ない（数行で完了） |

### いつどちらを使うか？

- **FoundryMemoryProvider**: 素早くメモリ機能を追加したい場合、Azure AI Foundry をすでに使っている場合
- **カスタム BaseContextProvider**: メモリの構造を完全に制御したい場合、Foundry 以外のストレージを使いたい場合